# Tweet Topic Modelling and Virality Prediction

# 🧠 BERTopic + Tabular Neural Network Pipeline 

End-to-end pipeline with train/test split, BERTopic topic modeling, and a Tabular NN classifier.

**Pipeline overview:**
1. Load & clean data
2. Train/Test split
3. Compute embeddings
4. BERTopic on training data only
5. Build **Tabular Neural Network** on BERTopic topic vectors
6. Save BERTopic + Tabular NN models
7. Testing: retweet prediction & class prediction

**Tabular NN design:**
- Dedicated **feature tokenizer** (linear projection per feature → d_token)
- **Gated residual blocks** (skip connections + gating for tabular data)
- **Ghost Batch Normalisation** (handles small effective batch sizes)
- Separate regression and classification heads
- Dropout + weight decay for regularisation


## 📂 1. Load & Label Data

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime
from tabulate import tabulate

DB_PATH = "../data/tweetsCleanedDB.sqlite"
TABLE_NAME = "tweets"

conn = sqlite3.connect(DB_PATH)
print(f"Processing table: {TABLE_NAME}")

df = pd.read_sql(f"SELECT * FROM {TABLE_NAME}", conn)
print(f"Loaded {len(df)} rows")

if 'Timestamp' in df.columns:
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')

print(df.head())
print(df.dtypes)


Processing table: tweets
Loaded 9909 rows
  tweet_id        username                                               text  \
0        1         julie81  Party least receive say or single. Prevent pre...   
1        2   richardhester  Hotel still Congress may member staff. Media d...   
2        3  williamsjoseph  Nice be her debate industry that year. Film wh...   
3        4     danielsmary  Laugh explain situation career occur serious. ...   
4        5      carlwarren  Involve sense former often approach government...   

   retweets  likes            timestamp        date      time  hour  \
0         2     25  2023-01-30 11:00:51  2023-01-30  11:00:51    11   
1        35     29  2023-01-02 22:45:58  2023-01-02  22:45:58    22   
2        51     25  2023-01-18 11:25:19  2023-01-18  11:25:19    11   
3        37     18  2023-04-10 22:06:29  2023-04-10  22:06:29    22   
4        27     80  2023-01-24 07:12:21  2023-01-24  07:12:21     7   

    day_name year_week year_month  year  
0 

## 📂 2. Label Data (Retweet Quartile Classes)

In [2]:
# Calculate quartile boundaries
q25 = df['retweets'].quantile(0.25)
q50 = df['retweets'].quantile(0.50)
q75 = df['retweets'].quantile(0.75)

print("Retweet Quartile Boundaries:")
print(f"  Min:      {df['retweets'].min()}")
print(f"  Q1 (25%): {q25}")
print(f"  Q2 (50%): {q50}")
print(f"  Q3 (75%): {q75}")
print(f"  Max:      {df['retweets'].max()}")

# Assign numeric label (0-3) based on quartiles
df['popularity_label'] = pd.cut(
    df['retweets'],
    bins=[-np.inf, q25, q50, q75, np.inf],
    labels=[0, 1, 2, 3]
).astype(int)

print("\nClass distribution:")
print(df['popularity_label'].value_counts().sort_index())


Retweet Quartile Boundaries:
  Min:      0
  Q1 (25%): 25.0
  Q2 (50%): 49.0
  Q3 (75%): 75.0
  Max:      100

Class distribution:
popularity_label
0    2548
1    2407
2    2567
3    2387
Name: count, dtype: int64


## 📦 3. Imports & Setup

In [3]:
import os
import re
import math
import numpy as np
import pandas as pd
import torch
import joblib

from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance

# Tabular NN
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, classification_report, mean_absolute_error

print('✅ All imports successful.')


I0000 00:00:1775046833.028960   20741 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1775046834.303255   20741 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775046845.756043   20741 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


✅ All imports successful.


## 🧹 4. Text Cleaning

In [4]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)           # remove URLs
    text = re.sub(r'@\w+', '', text)                      # remove mentions
    text = re.sub(r'#(\w+)', r'\1', text)                # strip # but keep word
    text = re.sub(r'^rt\s+', '', text, flags=re.MULTILINE)# remove RT prefix
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)           # remove special chars
    text = re.sub(r'\s+', ' ', text).strip()              # collapse whitespace
    return text

TEXT_COL = 'text' if 'text' in df.columns else df.columns[0]
print(f"Using text column: '{TEXT_COL}'")

df['text_clean'] = df[TEXT_COL].fillna('').apply(clean_text)
df = df[df['text_clean'].str.len() > 10].reset_index(drop=True)
print(f"Documents after cleaning: {len(df):,}")
print(df['text_clean'].head(3).tolist())


Using text column: 'text'
Documents after cleaning: 9,909
['party least receive say or single prevent prevent husband affect may himself cup style evening protect effect another themselves stage perform possible try tax share style television with successful much sell development economy effect', 'hotel still congress may member staff media draw buy fly identify on another turn minute would local subject way believe which question some message own all imagine join agency indicate', 'nice be her debate industry that year film where generation push discover partner level nearly money store style may enjoy kid discuss blue save model another about along everybody especially dinner character yard']


## ✂️ 5. Train / Test Split

Split is performed **immediately after loading and cleaning**, before any topic modeling.
BERTopic is then fitted on **training data only** to avoid data leakage.


In [5]:
TEST_SIZE = 0.2
RANDOM_STATE = 42

df_train, df_test = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df['popularity_label']  # preserve class balance
)

df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f"Train size: {len(df_train):,}  ({len(df_train)/len(df)*100:.1f}%)")
print(f"Test  size: {len(df_test):,}  ({len(df_test)/len(df)*100:.1f}%)")
print("\nTrain class distribution:")
print(df_train['popularity_label'].value_counts().sort_index())
print("\nTest class distribution:")
print(df_test['popularity_label'].value_counts().sort_index())

# Convenience lists
docs_train = df_train['text_clean'].tolist()
docs_test  = df_test['text_clean'].tolist()


Train size: 7,927  (80.0%)
Test  size: 1,982  (20.0%)

Train class distribution:
popularity_label
0    2038
1    1926
2    2053
3    1910
Name: count, dtype: int64

Test class distribution:
popularity_label
0    510
1    481
2    514
3    477
Name: count, dtype: int64


## 🔡 6. Embedding Model

In [6]:
EMBED_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'▶ Using device: {device}')

embedding_model = SentenceTransformer(EMBED_MODEL, device=device)
if device == 'cuda':
    embedding_model = embedding_model.half()  # FP16


▶ Using device: cpu


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 🔢 7. Compute & Cache Embeddings (Train & Test)

In [7]:
CACHE_TRAIN = 'embeddings_train.npy'
CACHE_TEST  = 'embeddings_test.npy'

def get_embeddings(docs, cache_path, model, device):
    if os.path.exists(cache_path):
        print(f"Loading cached embeddings from '{cache_path}'...")
        emb = np.load(cache_path)
        print(f'✅ Loaded from cache — shape: {emb.shape}')
    else:
        print(f'Computing embeddings for {len(docs):,} documents...')
        batch_size = 256 if device == 'cuda' else (64 if device == 'mps' else 32)
        emb = model.encode(
            docs,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )
        np.save(cache_path, emb)
        print(f'✅ Embeddings saved to {cache_path} — shape: {emb.shape}')
    return emb

embeddings_train = get_embeddings(docs_train, CACHE_TRAIN, embedding_model, device)
embeddings_test  = get_embeddings(docs_test,  CACHE_TEST,  embedding_model, device)


Loading cached embeddings from 'embeddings_train.npy'...
✅ Loaded from cache — shape: (7927, 384)
Loading cached embeddings from 'embeddings_test.npy'...
✅ Loaded from cache — shape: (1982, 384)


## 🏋️ 8. BERTopic Training

BERTopic is trained exclusively on the training split.
Topics are then **inferred** (transform) for the test split.


In [8]:
# ── Representation models ──────────────────────────────────────────────────
keybert_model = KeyBERTInspired(top_n_words=5)
mmr_model     = MaximalMarginalRelevance(diversity=0.3, top_n_words=5)

fast_representation = {
    'KeyBERT': keybert_model,
    'MMR':     mmr_model,
}

# ── Sub-models ─────────────────────────────────────────────────────────────
umap_model = UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0,
    metric='cosine', low_memory=True, random_state=42, n_jobs=-1,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=10, metric='euclidean',
    cluster_selection_method='eom', prediction_data=True,
    approx_min_span_tree=True, core_dist_n_jobs=-1,
)

vectorizer_model = CountVectorizer(
    max_features=10_000, min_df=3, ngram_range=(1, 2), strip_accents='unicode',
)

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

# ── Retweet-based engagement weight (train only) ───────────────────────────
df_train['engagement_weight'] = np.log1p(df_train['retweets'])
df_train['engagement_weight'] = (
    df_train['engagement_weight'] - df_train['engagement_weight'].min()
) / (df_train['engagement_weight'].max() - df_train['engagement_weight'].min())

# ── BERTopic model ─────────────────────────────────────────────────────────
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=fast_representation,
    calculate_probabilities=True,
    verbose=True,
)

# ── Fit on TRAIN only ──────────────────────────────────────────────────────
print("Fitting BERTopic on training data...")
train_topics, train_probs = topic_model.fit_transform(
    docs_train,
    embeddings=embeddings_train,
    y=df_train['engagement_weight'].values,
)

df_train['topic_id'] = train_topics
print(f"\n✅ BERTopic fit complete. Topics found: {topic_model.get_topic_info().shape[0] - 1}")
print(topic_model.get_topic_info().head(10))


Fitting BERTopic on training data...


2026-04-01 20:34:40,106 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-01 20:36:19,510 - BERTopic - Dimensionality - Completed ✓
2026-04-01 20:36:19,512 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-01 20:36:21,255 - BERTopic - Cluster - Completed ✓
2026-04-01 20:36:21,262 - BERTopic - Representation - Extracting topics from clusters using representation models.
2026-04-01 20:39:11,709 - BERTopic - Representation - Completed ✓



✅ BERTopic fit complete. Topics found: 44
   Topic  Count                                     Name  \
0     -1   5566                    -1_us_nice_agree_team   
1      0    286  0_republican_democrat_election_congress   
2      1    229           1_soldier_military_war_defense   
3      2    223           2_church_religious_forward_lot   
4      3    176               3_dog_animal_business_cost   
5      4    171             4_marriage_husband_wife_home   
6      5    153                   5_music_song_sing_rock   
7      6    105                      6_win_cup_page_song   
8      7     88            7_drug_medical_health_imagine   
9      8     85                 8_water_push_lose_decide   

                                      Representation  \
0  [us, nice, agree, team, message, son, country,...   
1  [republican, democrat, election, congress, pol...   
2  [soldier, military, war, defense, gun, under, ...   
3  [church, religious, forward, lot, pm, major, s...   
4  [dog, animal,

## 🔄 9. Transform Test Data with Fitted BERTopic

In [9]:
print("Transforming test data with fitted BERTopic...")
test_topics, test_probs = topic_model.transform(
    docs_test,
    embeddings=embeddings_test,
)

df_test['topic_id'] = test_topics
print(f"✅ Test transform complete. Unique test topics: {len(set(test_topics))}")
print(pd.Series(test_topics).value_counts().head(10))


2026-04-01 20:39:12,016 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.


Transforming test data with fitted BERTopic...


2026-04-01 20:39:38,689 - BERTopic - Dimensionality - Completed ✓
2026-04-01 20:39:38,690 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-01 20:39:38,865 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-04-01 20:39:39,276 - BERTopic - Probabilities - Completed ✓
2026-04-01 20:39:39,278 - BERTopic - Cluster - Completed ✓


✅ Test transform complete. Unique test topics: 16
-1     1825
 2       29
 1       24
 3       23
 4       16
 8       13
 7       13
 11      10
 9        8
 0        5
Name: count, dtype: int64


## 🧮 10. Build Feature Vectors for Tabular NN

Use **topic probability distributions** (soft topic assignments) as input features to the Tabular NN model.
These vectors capture which topics a tweet belongs to and how strongly.


In [10]:
def get_topic_features(probs, n_topics):
    """Ensure consistent feature matrix shape."""
    if probs is None or (hasattr(probs, 'ndim') and probs.ndim == 1):
        # single sample or no probabilities — return zeros
        return np.zeros((1, n_topics))
    if hasattr(probs, 'shape'):
        arr = np.array(probs)
    else:
        arr = np.array(list(probs))
    # pad/trim to n_topics columns
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
    if arr.shape[1] < n_topics:
        arr = np.pad(arr, ((0, 0), (0, n_topics - arr.shape[1])))
    elif arr.shape[1] > n_topics:
        arr = arr[:, :n_topics]
    return arr

n_topics = len(topic_model.get_topic_info()) - 1  # exclude -1 (outliers)
print(f"Number of topics (features): {n_topics}")

X_train = get_topic_features(train_probs, n_topics)
X_test  = get_topic_features(test_probs,  n_topics)

y_retweet_train = df_train['retweets'].values.astype(np.float32)
y_retweet_test  = df_test['retweets'].values.astype(np.float32)

y_class_train = df_train['popularity_label'].values.astype(np.int64)
y_class_test  = df_test['popularity_label'].values.astype(np.int64)

print(f"X_train shape: {X_train.shape}")
print(f"X_test  shape: {X_test.shape}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled  = scaler.transform(X_test).astype(np.float32)

# Log-transform retweets for regression (smoother target)
y_log_train = np.log1p(y_retweet_train)
y_log_test  = np.log1p(y_retweet_test)

print("\n✅ Feature matrices ready.")
print(f"Retweet range (train): {y_retweet_train.min():.0f} – {y_retweet_train.max():.0f}")
print(f"Class distribution (train):", np.bincount(y_class_train))


Number of topics (features): 44
X_train shape: (7927, 44)
X_test  shape: (1982, 44)

✅ Feature matrices ready.
Retweet range (train): 0 – 100
Class distribution (train): [2038 1926 2053 1910]


## 🤖 11. Tabular Neural Network Model

A purpose-built **Tabular NN** that replaces the generic MLP with tabular-specific components:

| Component | Purpose |
|---|---|
| **Feature Tokenizer** | Projects each topic probability into a d_token-dim representation |
| **Gated Residual Block** | Skip connection + sigmoid gate — better gradient flow for tabular data |
| **Ghost Batch Norm** | Splits batch into virtual sub-batches to stabilise BN stats |
| **Dual Head** | Regression (retweet count) + Classification (popularity class) |

Both outputs share a common trunk trained on BERTopic topic-probability vectors.


In [11]:
# ──────────────────────────────────────────────────────────────────────────
#  Tabular NN Building Blocks
# ──────────────────────────────────────────────────────────────────────────

class GhostBatchNorm(nn.Module):
    """
    Ghost Batch Normalisation — splits the batch into virtual sub-batches
    of size `virtual_batch_size` before computing BN statistics.
    This improves generalisation when true batch sizes are large.
    """
    def __init__(self, num_features: int, virtual_batch_size: int = 64, momentum: float = 0.02):
        super().__init__()
        self.virtual_batch_size = virtual_batch_size
        self.bn = nn.BatchNorm1d(num_features, momentum=momentum)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or x.size(0) <= self.virtual_batch_size:
            return self.bn(x)
        # Split into virtual sub-batches
        chunks = x.split(self.virtual_batch_size, dim=0)
        return torch.cat([self.bn(c) for c in chunks], dim=0)


class GatedResidualBlock(nn.Module):
    """
    Gated Residual Block for tabular data.

    Architecture:
        x  →  Linear → GhostBN → ELU  →  Linear → GhostBN
           →  gate (sigmoid)  →  element-wise multiply
           →  residual add (with optional projection)
           →  ELU
    """
    def __init__(
        self,
        in_dim: int,
        out_dim: int,
        dropout: float = 0.3,
        virtual_batch_size: int = 64,
    ):
        super().__init__()
        self.fc1   = nn.Linear(in_dim,  out_dim)
        self.bn1   = GhostBatchNorm(out_dim, virtual_batch_size)
        self.fc2   = nn.Linear(out_dim, out_dim)
        self.bn2   = GhostBatchNorm(out_dim, virtual_batch_size)
        self.gate  = nn.Linear(out_dim, out_dim)   # sigmoid gate
        self.drop  = nn.Dropout(dropout)

        # project residual if dims differ
        self.proj = (
            nn.Linear(in_dim, out_dim, bias=False)
            if in_dim != out_dim else nn.Identity()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = F.elu(self.bn1(self.fc1(x)))
        h = self.drop(h)
        h = self.bn2(self.fc2(h))
        g = torch.sigmoid(self.gate(h))   # gating
        h = h * g                         # element-wise gate
        return F.elu(h + self.proj(x))    # skip connection


class FeatureTokenizer(nn.Module):
    """
    Projects each input feature independently into a d_token-dimensional space,
    similar to the FT-Transformer approach. This lets each topic probability
    have its own dedicated representation before mixing.
    """
    def __init__(self, n_features: int, d_token: int):
        super().__init__()
        # Weight: (n_features, d_token),  Bias: (n_features, d_token)
        self.weight = nn.Parameter(torch.randn(n_features, d_token) * 0.02)
        self.bias   = nn.Parameter(torch.zeros(n_features, d_token))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, n_features)  →  (B, n_features * d_token)
        # Broadcasting: x.unsqueeze(-1) * weight  →  (B, n_features, d_token)
        tokens = x.unsqueeze(-1) * self.weight + self.bias   # (B, F, d_token)
        return tokens.flatten(1)                              # (B, F * d_token)


# ──────────────────────────────────────────────────────────────────────────
#  Main Tabular NN
# ──────────────────────────────────────────────────────────────────────────

class TabularNN(nn.Module):
    """
    Multi-task Tabular Neural Network.

    Architecture:
        Input (topic probs)
          │
          ├── FeatureTokenizer  (per-feature linear projection to d_token dims)
          │
          ├── GatedResidualBlock  × n_blocks  (shared trunk)
          │
          ├── Regression head  → predicted log-retweet count
          └── Classification head  → class logits (4 classes)
    """
    def __init__(
        self,
        n_features:          int,
        n_classes:           int   = 4,
        d_token:             int   = 8,
        hidden_dim:          int   = 256,
        n_blocks:            int   = 4,
        dropout:             float = 0.3,
        virtual_batch_size:  int   = 64,
    ):
        super().__init__()

        tokenizer_out = n_features * d_token

        self.tokenizer = FeatureTokenizer(n_features, d_token)

        # Build trunk: first block handles dimension change, rest are stable
        blocks = [GatedResidualBlock(tokenizer_out, hidden_dim, dropout, virtual_batch_size)]
        for _ in range(n_blocks - 1):
            blocks.append(GatedResidualBlock(hidden_dim, hidden_dim, dropout, virtual_batch_size))
        self.trunk = nn.Sequential(*blocks)

        # Task heads
        self.reg_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, 1),
        )
        self.cls_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, n_classes),
        )

    def forward(self, x: torch.Tensor):
        tokens = self.tokenizer(x)          # feature tokenization
        h = self.trunk(tokens)              # gated residual trunk
        retweet_pred = self.reg_head(h).squeeze(-1)
        class_logits = self.cls_head(h)
        return retweet_pred, class_logits


# ──────────────────────────────────────────────────────────────────────────
#  Training loop
# ──────────────────────────────────────────────────────────────────────────

def train_tabular_nn(
    X_tr, y_log_tr, y_cls_tr,
    X_va, y_log_va, y_cls_va,
    n_classes=4,
    epochs=60,
    batch_size=256,
    lr=1e-3,
    d_token=8,
    hidden_dim=256,
    n_blocks=4,
    dropout=0.3,
    virtual_batch_size=64,
    device='cpu',
):
    n_features = X_tr.shape[1]
    model = TabularNN(
        n_features=n_features,
        n_classes=n_classes,
        d_token=d_token,
        hidden_dim=hidden_dim,
        n_blocks=n_blocks,
        dropout=dropout,
        virtual_batch_size=virtual_batch_size,
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Tabular NN parameters: {total_params:,}")

    opt       = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr * 0.01)
    mse_loss  = nn.MSELoss()
    ce_loss   = nn.CrossEntropyLoss(label_smoothing=0.05)  # label smoothing for cls

    def to_tensor(arr): return torch.tensor(arr)
    tr_ds = TensorDataset(to_tensor(X_tr),
                          to_tensor(y_log_tr),
                          to_tensor(y_cls_tr))
    va_ds = TensorDataset(to_tensor(X_va),
                          to_tensor(y_log_va),
                          to_tensor(y_cls_va))
    tr_dl = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,  drop_last=True)
    va_dl = DataLoader(va_ds, batch_size=512)

    best_val_loss, best_state = float('inf'), None

    for epoch in range(1, epochs + 1):
        # ── Train ──────────────────────────────────────────────────────────
        model.train()
        for Xb, yb_reg, yb_cls in tr_dl:
            Xb      = Xb.to(device)
            yb_reg  = yb_reg.to(device)
            yb_cls  = yb_cls.to(device)
            opt.zero_grad()
            pred_reg, pred_cls = model(Xb)
            loss = mse_loss(pred_reg, yb_reg) + ce_loss(pred_cls, yb_cls)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
            opt.step()

        scheduler.step()

        # ── Validate ───────────────────────────────────────────────────────
        model.eval()
        val_losses = []
        with torch.no_grad():
            for Xb, yb_reg, yb_cls in va_dl:
                Xb, yb_reg, yb_cls = Xb.to(device), yb_reg.to(device), yb_cls.to(device)
                pred_reg, pred_cls = model(Xb)
                vloss = mse_loss(pred_reg, yb_reg) + ce_loss(pred_cls, yb_cls)
                val_losses.append(vloss.item())
        val_loss = np.mean(val_losses)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if epoch % 10 == 0 or epoch == 1:
            lr_now = scheduler.get_last_lr()[0]
            print(f"Epoch {epoch:>3} | val_loss: {val_loss:.4f} | lr: {lr_now:.2e}")

    model.load_state_dict(best_state)
    print(f"\n✅ Training complete. Best val loss: {best_val_loss:.4f}")
    return model


# ── Use a 10% validation split from train ─────────────────────────────────
val_frac = 0.1
n_val    = max(1, int(len(X_train_scaled) * val_frac))
X_val_s, y_log_val, y_cls_val = (
    X_train_scaled[-n_val:], y_log_train[-n_val:], y_class_train[-n_val:]
)
X_tr_s, y_log_tr, y_cls_tr = (
    X_train_scaled[:-n_val], y_log_train[:-n_val], y_class_train[:-n_val]
)

tab_device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Training Tabular NN on device: {tab_device}")
print(f"Architecture: FeatureTokenizer(d_token=8) → 4× GatedResidualBlock(256) → dual heads")
print()

tab_model = train_tabular_nn(
    X_tr_s, y_log_tr, y_cls_tr,
    X_val_s, y_log_val, y_cls_val,
    n_classes=4,
    epochs=60,
    batch_size=256,
    lr=1e-3,
    d_token=8,
    hidden_dim=256,
    n_blocks=4,
    dropout=0.3,
    virtual_batch_size=64,
    device=tab_device,
)


Training Tabular NN on device: cpu
Architecture: FeatureTokenizer(d_token=8) → 4× GatedResidualBlock(256) → dual heads

  Tabular NN parameters: 975,429
Epoch   1 | val_loss: 3.1374 | lr: 9.99e-04
Epoch  10 | val_loss: 2.1551 | lr: 9.34e-04
Epoch  20 | val_loss: 2.1573 | lr: 7.53e-04
Epoch  30 | val_loss: 2.1694 | lr: 5.05e-04
Epoch  40 | val_loss: 2.1615 | lr: 2.58e-04
Epoch  50 | val_loss: 2.1558 | lr: 7.63e-05
Epoch  60 | val_loss: 2.1610 | lr: 1.00e-05

✅ Training complete. Best val loss: 2.1281


## 💾 12. Save BERTopic + Tabular NN Models

In [12]:
import pickle

BERTOPIC_PATH  = '../data/bertopic_model'
TAB_MODEL_PATH = '../data/tabular_nn_model.pt'
SCALER_PATH    = '../data/topic_scaler.pkl'
META_PATH      = '../data/pipeline_meta.pkl'

os.makedirs('../data', exist_ok=True)

# ── Save BERTopic ──────────────────────────────────────────────────────────
topic_model.save(
    BERTOPIC_PATH,
    serialization='pickle',
    save_ctfidf=True,
    save_embedding_model=EMBED_MODEL,
)
print(f"✅ BERTopic saved to '{BERTOPIC_PATH}'")

# ── Save Tabular NN model ──────────────────────────────────────────────────
torch.save({
    'model_state_dict': tab_model.state_dict(),
    'n_features':       X_train_scaled.shape[1],
    'n_classes':        4,
    'n_topics':         n_topics,
    'd_token':          8,
    'hidden_dim':       256,
    'n_blocks':         4,
    'model_type':       'TabularNN',
}, TAB_MODEL_PATH)
print(f"✅ Tabular NN model saved to '{TAB_MODEL_PATH}'")

# ── Save scaler ────────────────────────────────────────────────────────────
joblib.dump(scaler, SCALER_PATH)
print(f"✅ Scaler saved to '{SCALER_PATH}'")

# ── Save pipeline meta (quartile thresholds) ───────────────────────────────
meta = {'q25': float(q25), 'q50': float(q50), 'q75': float(q75), 'n_topics': n_topics}
with open(META_PATH, 'wb') as f:
    pickle.dump(meta, f)
print(f"✅ Pipeline meta saved to '{META_PATH}'")


2026-04-01 20:54:11,636 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


✅ BERTopic saved to '../data/bertopic_model'
✅ Tabular NN model saved to '../data/tabular_nn_model.pt'
✅ Scaler saved to '../data/topic_scaler.pkl'
✅ Pipeline meta saved to '../data/pipeline_meta.pkl'


## 🔁 13. Full Pipeline (Train & Inference)

A unified `predict()` function that can be used on **new raw text** with identical preprocessing.
The same function is used for both training-time evaluation and test-time inference.


In [13]:
def pipeline_predict(texts, topic_model, embedding_model, tab_model, scaler,
                     n_topics, device='cpu'):
    """
    End-to-end inference pipeline:
        raw texts → clean → embed → BERTopic transform →
        scale → Tabular NN → (retweet_pred, class_pred)

    Returns
    -------
    retweet_preds : np.ndarray  (float32, original scale)
    class_preds   : np.ndarray  (int, 0-3)
    probs         : np.ndarray  (float32, class probabilities)
    """
    # 1. Clean
    cleaned = [clean_text(t) for t in texts]

    # 2. Embed
    embs = embedding_model.encode(
        cleaned, batch_size=64, normalize_embeddings=True,
        show_progress_bar=False, convert_to_numpy=True,
    )

    # 3. BERTopic transform
    topics, probs_bt = topic_model.transform(cleaned, embeddings=embs)
    X_feat = get_topic_features(probs_bt, n_topics)

    # 4. Scale
    X_sc = scaler.transform(X_feat).astype(np.float32)

    # 5. Tabular NN forward
    tab_model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X_sc).to(device)
        pred_log, logits = tab_model(X_t)
        pred_log   = pred_log.cpu().numpy()
        probs_cls  = torch.softmax(logits, dim=-1).cpu().numpy()
        class_preds = probs_cls.argmax(axis=1)

    # 6. Back-transform log retweets
    retweet_preds = np.expm1(pred_log)

    return retweet_preds, class_preds, probs_cls


print("✅ Pipeline predict function defined.")
print("Usage: retweet_preds, class_preds, probs = pipeline_predict(texts, ...)")


✅ Pipeline predict function defined.
Usage: retweet_preds, class_preds, probs = pipeline_predict(texts, ...)


## 🧪 14. Testing — Evaluate on Test Set

### 14a. Retweet Prediction (Regression)


In [14]:
# ── Predict on test set ───────────────────────────────────────────────────
tab_model.eval()
with torch.no_grad():
    X_test_t = torch.tensor(X_test_scaled).to(tab_device)
    pred_log_test, logits_test = tab_model(X_test_t)
    pred_log_test = pred_log_test.cpu().numpy()
    probs_test    = torch.softmax(logits_test, dim=-1).cpu().numpy()
    class_preds_test = probs_test.argmax(axis=1)

retweet_preds_test = np.expm1(pred_log_test)
retweet_true_test  = y_retweet_test

# ── Regression metrics ─────────────────────────────────────────────────────
rmse = math.sqrt(mean_squared_error(retweet_true_test, retweet_preds_test))
mae  = mean_absolute_error(retweet_true_test, retweet_preds_test)

# Log-scale metrics (less dominated by outliers)
rmse_log = math.sqrt(mean_squared_error(np.log1p(retweet_true_test), np.log1p(retweet_preds_test)))
mae_log  = mean_absolute_error(np.log1p(retweet_true_test), np.log1p(retweet_preds_test))

print("=" * 50)
print("  RETWEET PREDICTION — TEST SET METRICS")
print("=" * 50)
print(f"  RMSE        (raw scale): {rmse:>12,.2f}")
print(f"  MAE         (raw scale): {mae:>12,.2f}")
print(f"  RMSE (log1p scale): {rmse_log:.4f}")
print(f"  MAE  (log1p scale): {mae_log:.4f}")
print("=" * 50)

# ── Sample predictions ─────────────────────────────────────────────────────
sample_df = pd.DataFrame({
    'text':          df_test['text_clean'].head(10),
    'true_retweets': retweet_true_test[:10],
    'pred_retweets': retweet_preds_test[:10].round().astype(int),
    'topic_id':      df_test['topic_id'].head(10),
})
print("\nSample predictions (first 10 test documents):")
print(sample_df.to_string(index=False))


  RETWEET PREDICTION — TEST SET METRICS
  RMSE        (raw scale):        31.83
  MAE         (raw scale):        26.75
  RMSE (log1p scale): 0.9484
  MAE  (log1p scale): 0.7083

Sample predictions (first 10 test documents):
                                                                                                                                                                                                                                                         text  true_retweets  pred_retweets  topic_id
                                                 election situation security tell special fact technology crime mean central cost exist field travel network modern audience after evidence chance in heart believe street who lead heart look almost mention           72.0             43        -1
        gun success build question special six eye machine myself then anyone thousand provide man son section car lot history each reflect force government company eye get second skin ne

### 14b. Class Prediction (Popularity Label)

In [15]:
print("=" * 55)
print("  POPULARITY CLASS PREDICTION — TEST SET METRICS")
print("=" * 55)
print(classification_report(
    y_class_test,
    class_preds_test,
    target_names=['Q1 (low)', 'Q2', 'Q3', 'Q4 (viral)'],
    digits=4,
))

# ── Confusion-style breakdown ──────────────────────────────────────────────
from sklearn.metrics import confusion_matrix, accuracy_score

acc = accuracy_score(y_class_test, class_preds_test)
print(f"Overall accuracy: {acc:.4f}")

cm = confusion_matrix(y_class_test, class_preds_test)
cm_df = pd.DataFrame(
    cm,
    index=['True Q1', 'True Q2', 'True Q3', 'True Q4'],
    columns=['Pred Q1', 'Pred Q2', 'Pred Q3', 'Pred Q4'],
)
print("\nConfusion Matrix:")
print(cm_df.to_string())


  POPULARITY CLASS PREDICTION — TEST SET METRICS
              precision    recall  f1-score   support

    Q1 (low)     0.2552    0.7451    0.3802       510
          Q2     0.1923    0.0104    0.0197       481
          Q3     0.2633    0.1926    0.2225       514
  Q4 (viral)     0.1868    0.0356    0.0599       477

    accuracy                         0.2528      1982
   macro avg     0.2244    0.2459    0.1706      1982
weighted avg     0.2256    0.2528    0.1747      1982

Overall accuracy: 0.2528

Confusion Matrix:
         Pred Q1  Pred Q2  Pred Q3  Pred Q4
True Q1      380       12       90       28
True Q2      359        5       90       27
True Q3      390        6       99       19
True Q4      360        3       97       17


### 14c. End-to-End Pipeline Demo (new raw texts)

In [16]:
sample_texts = [
    "Breaking: major policy announcement from the White House today #politics",
    "Just had the best coffee ever at the local cafe ☕",
    "New study links social media use to anxiety in teens",
    "HUGE news: Scientists discover potential cure for common cold!",
]

rt_preds, cls_preds, cls_probs = pipeline_predict(
    sample_texts, topic_model, embedding_model, tab_model, scaler,
    n_topics=n_topics, device=tab_device
)

class_labels = ['Q1 (low)', 'Q2', 'Q3', 'Q4 (viral)']

print("\n📊 End-to-End Pipeline Predictions:")
print("-" * 80)
for text, rt, cls, prob in zip(sample_texts, rt_preds, cls_preds, cls_probs):
    print(f"Text:        {text[:60]}...")
    print(f"  Predicted retweets: {rt:.1f}")
    print(f"  Predicted class:    {class_labels[cls]}")
    print(f"  Class probs:        {dict(zip(class_labels, prob.round(3)))}")
    print()


2026-04-01 20:54:15,686 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-04-01 20:54:15,733 - BERTopic - Dimensionality - Completed ✓
2026-04-01 20:54:15,736 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-01 20:54:15,743 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-04-01 20:54:15,759 - BERTopic - Probabilities - Completed ✓
2026-04-01 20:54:15,761 - BERTopic - Cluster - Completed ✓



📊 End-to-End Pipeline Predictions:
--------------------------------------------------------------------------------
Text:        Breaking: major policy announcement from the White House tod...
  Predicted retweets: 42.4
  Predicted class:    Q3
  Class probs:        {'Q1 (low)': np.float32(0.228), 'Q2': np.float32(0.229), 'Q3': np.float32(0.285), 'Q4 (viral)': np.float32(0.258)}

Text:        Just had the best coffee ever at the local cafe ☕...
  Predicted retweets: 39.7
  Predicted class:    Q1 (low)
  Class probs:        {'Q1 (low)': np.float32(0.287), 'Q2': np.float32(0.22), 'Q3': np.float32(0.27), 'Q4 (viral)': np.float32(0.222)}

Text:        New study links social media use to anxiety in teens...
  Predicted retweets: 39.1
  Predicted class:    Q1 (low)
  Class probs:        {'Q1 (low)': np.float32(0.293), 'Q2': np.float32(0.22), 'Q3': np.float32(0.264), 'Q4 (viral)': np.float32(0.224)}

Text:        HUGE news: Scientists discover potential cure for common col...
  Predicted ret

## 📊 15. Topic Analysis & Visualisations

In [17]:
# ── Topic summary with retweet stats (train set) ──────────────────────────
topic_info = topic_model.get_topic_info()

retweet_by_topic = (
    df_train.groupby('topic_id')['retweets']
    .agg(avg_retweets='mean', total_retweets='sum', doc_count='count')
    .reset_index()
    .rename(columns={'topic_id': 'Topic'})
)

topic_info = topic_info.merge(retweet_by_topic, on='Topic', how='left')
topic_info = topic_info.sort_values('avg_retweets', ascending=False)

print(f"Total topics: {len(topic_info)}")
print(topic_info[['Topic', 'Count', 'Name', 'avg_retweets', 'doc_count']].head(10).to_string(index=False))


Total topics: 45
 Topic  Count                                                          Name  avg_retweets  doc_count
    41     10            41_thus cup_threat we_read available_yard physical     67.800000         10
    35     12     35_doctor war_upon describe_the chair_authority treatment     65.833333         12
    42     10               42_oil_four always_write lead_democratic lawyer     63.100000         10
    37     12                    37_food_your near_upon describe_none learn     62.000000         12
    34     13           34_cold_yard physical_sell production_reveal market     61.076923         13
    40     10 40_television_drop south_environmental television_clear above     58.400000         10
    22     21                  22_training_throw think_could win_check miss     58.285714         21
    36     12            36_follow method_just hope_pm fact_interest finish     56.666667         12
    20     26                   20_color_your near_successful what_tree al

In [18]:
# ── Per-topic keywords ────────────────────────────────────────────────────
print('Top keywords per topic (top 5 by avg retweets):\n')
for topic_id in topic_info['Topic'].head(5):
    if topic_id == -1:
        continue
    words = topic_model.get_topic(topic_id)
    keywords = ', '.join([w for w, _ in words[:8]])
    avg_rt = topic_info.loc[topic_info['Topic'] == topic_id, 'avg_retweets'].values[0]
    print(f'  Topic {topic_id:>3} | avg retweets: {avg_rt:>8.1f} | {keywords}')


Top keywords per topic (top 5 by avg retweets):

  Topic  41 | avg retweets:     67.8 | thus cup, threat we, read available, yard physical, weight seem, executive, wife, man
  Topic  35 | avg retweets:     65.8 | doctor war, upon describe, the chair, authority treatment, treatment activity, doctor, hospital, whom
  Topic  42 | avg retweets:     63.1 | oil, four always, write lead, democratic lawyer, model, their, heart, customer
  Topic  37 | avg retweets:     62.0 | food, your near, upon describe, none learn, market anyone, nation, cultural, despite
  Topic  34 | avg retweets:     61.1 | cold, yard physical, sell production, reveal market, step certain, relate production, market relate, fight decide


In [19]:
# ── Visualisations ────────────────────────────────────────────────────────
topic_model.visualize_barchart(top_n_topics=10, n_words=8)


In [20]:
topic_model.visualize_heatmap(top_n_topics=20)


In [21]:
topic_model.visualize_topics()
